In [1]:
import sys
sys.path.append("/kaggle/input/datasets/risha8750/phase2-core3")
from phase2_core import *
import torch
import pandas as pd
from pathlib import Path

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cache_root = Path("/kaggle/input/notebooks/risha8750/major-project-phase-2a-cache/phase2_cache")
prev_outputs = Path("/kaggle/input/datasets/risha8750/phase2-latest-outputs/phase2_outputs")
corner_path = prev_outputs/"corner"/"corner_last.pt"
edge_path = prev_outputs/"edge"/"edge_last.pt"

assert cache_root.exists()
assert corner_path.exists()
assert edge_path.exists()

out_root = Path("/kaggle/working/phase2_outputs")
corner_dir = out_root/"corner"
edge_dir = out_root/"edge"
corner_dir.mkdir(parents=True, exist_ok=True)
edge_dir.mkdir(parents=True, exist_ok=True)

In [2]:
corner_state = torch.load(corner_path, map_location="cpu", weights_only=False)
edge_state = torch.load(edge_path, map_location="cpu", weights_only=False)

print("corner epochs done:", len(corner_state["history"]), "last epoch:", corner_state["epoch"])
print("edge epochs done:", len(edge_state["history"]), "last epoch:", edge_state["epoch"])

corner epochs done: 20 last epoch: 19
edge epochs done: 8 last epoch: 7


In [3]:
train_set = RoofDataset(cache_root, "train", augment=True)
val_set = RoofDataset(cache_root, "val", augment=False)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=8, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=8, shuffle=False, num_workers=2)

assert len(train_set) > 0
assert len(val_set) > 0

In [4]:
corner_model = CornerPointNet2().to(device)
corner_model.load_state_dict(corner_state["model"])

for parameter in corner_model.parameters():
    parameter.requires_grad = False
corner_model.eval()

assert next(corner_model.parameters()).requires_grad == False
assert corner_state["epoch"] == 19

In [5]:
edge_model = EdgeGraphNet().to(device)
edge_model.load_state_dict(edge_state["model"])

edge_optimizer = torch.optim.Adam(edge_model.parameters(), lr=1e-4)
edge_optimizer.load_state_dict(edge_state["optimizer"])

edge_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(edge_optimizer, T_max=20)
edge_scheduler.load_state_dict(edge_state["scheduler"])

edge_history = edge_state["history"]
edge_start = edge_history[-1]["epoch"] + 1

assert edge_start == 8
print(edge_start, edge_history[-1])

8 {'epoch': 7, 'train_loss': 0.0167485819844415, 'val_loss': 0.016266808015138662, 'lr': 8.511087728614862e-05}


In [6]:
def edge_epoch(loader, train):
    edge_model.train() if train else edge_model.eval()

    total = 0.0
    count = 0
    total_batches = len(loader)

    mode = "Training" if train else "Validation"

    print(f"\n{mode} started | Total batches: {total_batches}")

    for batch_idx, batch in enumerate(loader):
        points = batch["points"].to(device)
        corners = batch["corners"].to(device)
        corner_count = batch["corner_count"].to(device)
        edges = batch["edges"].to(device)
        edge_count = batch["edge_count"].to(device)

        with torch.no_grad():
            logits, offsets, shared = corner_model(points)

        batch_loss = 0.0
        valid_samples = 0

        for i in range(points.shape[0]):
            n = int(corner_count[i])

            if n < 2:
                continue

            pred_corners, conf = predict_corners(
                points[i],
                logits[i],
                offsets[i],
                batch["scale"][i]
            )

            if len(pred_corners) < 2:
                continue

            feat = interpolate_features(
                points[i],
                shared[i],
                pred_corners
            )

            corner_xyz = torch.as_tensor(
                pred_corners,
                dtype=points.dtype,
                device=device
            )

            corner_scores = torch.as_tensor(
                conf,
                dtype=points.dtype,
                device=device
            )

            pairs, edge_logits = edge_model(
                corner_xyz,
                corner_scores,
                feat
            )

            labels = edge_labels(
                pairs,
                edges[i],
                edge_count[i]
            )

            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                edge_logits,
                labels
            )

            batch_loss = batch_loss + loss
            valid_samples += 1

        if train and valid_samples > 0:
            edge_optimizer.zero_grad()
            batch_loss.backward()
            edge_optimizer.step()

        batch_loss_value = float(batch_loss.detach())

        total += batch_loss_value
        count += 1

        # Print every batch
        print(
            f"\r{mode} Batch [{batch_idx + 1}/{total_batches}] "
            f"Loss: {batch_loss_value:.6f} "
            f"Valid Samples: {valid_samples}",
            end="",
            flush=True
        )

    print()

    return total / max(count, 1)

In [7]:
edge_epochs_this_run = 12
edge_target_epochs = 20
edge_stop = min(edge_start + edge_epochs_this_run, edge_target_epochs)

print(f"Starting training from epoch {edge_start} to {edge_stop - 1}")
print(f"Total epochs this run: {edge_stop - edge_start}")

assert edge_stop == 20

for epoch in range(edge_start, edge_stop):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{edge_target_epochs}")
    print(f"{'='*50}")

    train_loss = edge_epoch(train_loader, True)
    print(f"Train Loss: {train_loss:.6f}")

    val_loss = edge_epoch(val_loader, False)
    print(f"Validation Loss: {val_loss:.6f}")

    assert np.isfinite(train_loss)
    assert np.isfinite(val_loss)

    edge_scheduler.step()

    current_lr = edge_optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": current_lr
    }

    edge_history.append(row)

    state = {
        "epoch": epoch,
        "model": edge_model.state_dict(),
        "optimizer": edge_optimizer.state_dict(),
        "scheduler": edge_scheduler.state_dict(),
        "history": edge_history
    }

    torch.save(state, edge_dir / "edge_last.pt")

    best_val_loss = min(x["val_loss"] for x in edge_history)

    if val_loss <= best_val_loss:
        torch.save(state, edge_dir / "edge_best.pt")
        print(f"Learning Rate: {current_lr:.2e}")
        print("NEW BEST MODEL SAVED")
    else:
        print(f"Learning Rate: {current_lr:.2e}")

    print(f"Last checkpoint saved: {edge_dir / 'edge_last.pt'}")

assert (edge_dir / "edge_last.pt").exists()
assert edge_history[-1]["epoch"] == 19

print("\nTraining completed successfully!")
print(f"Total epochs completed: {len(edge_history)}")
print(f"Best validation loss: {min(x['val_loss'] for x in edge_history):.6f}")

pd.DataFrame(edge_history).tail(20)

Starting training from epoch 8 to 19
Total epochs this run: 12

Epoch 9/20

Training started | Total batches: 3198
Training Batch [3198/3198] Loss: 1.850777 Valid Samples: 3
Train Loss: 4.693341

Validation started | Total batches: 397
Validation Batch [397/397] Loss: 1.227324 Valid Samples: 2
Validation Loss: 4.660922
Learning Rate: 8.15e-05
Last checkpoint saved: /kaggle/working/phase2_outputs/edge/edge_last.pt

Epoch 10/20

Training started | Total batches: 3198
Training Batch [3198/3198] Loss: 2.235106 Valid Samples: 3
Train Loss: 4.622338

Validation started | Total batches: 397
Validation Batch [397/397] Loss: 1.369483 Valid Samples: 2
Validation Loss: 4.627225
Learning Rate: 7.75e-05
Last checkpoint saved: /kaggle/working/phase2_outputs/edge/edge_last.pt

Epoch 11/20

Training started | Total batches: 3198
Training Batch [3198/3198] Loss: 1.305973 Valid Samples: 3
Train Loss: 4.599494

Validation started | Total batches: 397
Validation Batch [397/397] Loss: 1.183426 Valid Sample

,epoch,train_loss,val_loss,lr
0,0,0.041519,0.033337,0.000100
1,1,0.030726,0.028736,0.000099
2,2,0.026811,0.025170,0.000098
3,3,0.024264,0.023425,0.000096
4,4,0.022577,0.022270,0.000094
5,5,0.020841,0.019074,0.000091
6,6,0.018025,0.017545,0.000088
7,7,0.016749,0.016267,0.000085
8,8,4.693341,4.660922,0.000081
9,9,4.622338,4.627225,0.000077
